In [ ]:
!pip install transformers datasets seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report, f1_score

In [ ]:

!pip install datasets==2.14.6 fsspec==2023.6.0 huggingface_hub==0.20.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.13.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.20.3 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.6.0 which is incompatible

In [ ]:

dataset = load_dataset("xtreme", "udpos.English")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


README.md: 0.00B [00:00, ?B/s]

udpos.English/train-00000-of-00001.parqu(…):   0%|          | 0.00/1.48M [00:00<?, ?B/s]

udpos.English/validation-00000-of-00001.(…):   0%|          | 0.00/265k [00:00<?, ?B/s]

udpos.English/test-00000-of-00001.parque(…):   0%|          | 0.00/367k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21253 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3974 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 21253
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 3974
    })
    test: Dataset({
        features: ['tokens', 'pos_tags'],
        num_rows: 5440
    })
})


In [ ]:

example = dataset["train"][0]
print(example)

{'tokens': ['Aesthetic', 'Appreciation', 'and', 'Spanish', 'Art', ':'], 'pos_tags': [0, 7, 4, 0, 7, 12]}


In [ ]:
print(dataset["train"].features)

pos_feature = "pos_tags"   # <- use this
pos_label_list = dataset["train"].features[pos_feature].feature.names

id2label_pos = {i: l for i, l in enumerate(pos_label_list)}
label2id_pos = {l: i for i, l in enumerate(pos_label_list)}
num_pos_labels = len(pos_label_list)

print(f"Number of POS labels: {num_pos_labels}")
print("POS labels:", pos_label_list)

{'tokens': List(Value('string')), 'pos_tags': List(ClassLabel(names=['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']))}
Number of POS labels: 17
POS labels: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    # Tokenize list of tokens
    tokenized = tokenizer(
        examples["tokens"],              # list of words
        is_split_into_words=True,
        truncation=True,
        max_length=128,
    )

    all_labels = []

    for i, pos_tags in enumerate(examples[pos_feature]):   # pos_feature = "pos_tags"
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        labels = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens -> ignore
                labels.append(-100)
            elif word_id != prev_word_id:
                # First subword of a word -> real label
                labels.append(pos_tags[word_id])
            else:
                # Continuation subword -> ignore
                labels.append(-100)
            prev_word_id = word_id

        all_labels.append(labels)

    tokenized["labels"] = all_labels   # Trainer expects "labels"
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print("Tokenization & label alignment done.")
print("Features:", tokenized_dataset["train"].features)

Map:   0%|          | 0/3974 [00:00<?, ? examples/s]

Tokenization & label alignment done.
Features: {'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [ ]:
ex = tokenized_dataset["train"][0]
tokens = tokenizer.convert_ids_to_tokens(ex["input_ids"])

print(f"{'Token':<20} {'Label'}")
print("-" * 32)
for tok, lab in zip(tokens, ex["labels"]):
    label_str = id2label_pos[lab] if lab != -100 else "-100 (ignored)"
    print(f"{tok:<20} {label_str}")

Token                Label
--------------------------------
[CLS]                -100 (ignored)
A                    ADJ
##est                -100 (ignored)
##hetic              -100 (ignored)
A                    NOUN
##pp                 -100 (ignored)
##re                 -100 (ignored)
##ciation            -100 (ignored)
and                  CCONJ
Spanish              ADJ
Art                  NOUN
:                    PUNCT
[SEP]                -100 (ignored)


In [ ]:
from transformers import AutoModelForTokenClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_pos_labels,
    id2label=id2label_pos,
    label2id=label2id_pos,
)

model.to(device)
print("Model loaded on", device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Model loaded on cuda


In [ ]:
from transformers import DataCollatorForTokenClassification
import evaluate
import numpy as np

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_row, l_row = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_row.append(id2label_pos[p])
                l_row.append(id2label_pos[l])
        true_preds.append(p_row)
        true_labels.append(l_row)

    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": round(result["overall_precision"], 4),
        "recall":    round(result["overall_recall"],    4),
        "f1":        round(result["overall_f1"],        4),
        "accuracy":  round(result["overall_accuracy"],  4),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_pos_ud",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Step,Training Loss
50,1.114146
100,0.231141
150,0.182996
200,0.169311
250,0.165108
300,0.156880
350,0.168571
400,0.141865
450,0.142903
500,0.139190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3987, training_loss=0.09318808505252277, metrics={'train_runtime': 405.4921, 'train_samples_per_second': 157.239, 'train_steps_per_second': 9.832, 'total_flos': 1794168509716998.0, 'train_loss': 0.09318808505252277, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate(tokenized_dataset["test"])

print("Test set performance:")
for k, v in results.items():
    if k.startswith("eval_"):
        print(f"{k}: {v:.4f}")

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: 

Test set performance:
eval_loss: 0.1372
eval_precision: 0.9571
eval_recall: 0.9582
eval_f1: 0.9577
eval_accuracy: 0.9641
eval_runtime: 7.5975
eval_samples_per_second: 716.0240
eval_steps_per_second: 22.3760


In [ ]:
import pandas as pd

def predict(sentence: str) -> pd.DataFrame:
    model.eval()
    words = sentence.split()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoded)

    preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = encoded.word_ids(batch_index=0)

    rows, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            rows.append({
                "Token":  words[wid],
                "POS Tag": id2label_pos[preds[idx]],
            })
            seen.add(wid)

    return pd.DataFrame(rows)
# Example sentences
s1 = "John works at Google in UK"
print(f"Input: {s1}\n")
print(predict(s1).to_string(index=False))

s2 = "The quick brown fox jumps over the lazy dog"
print(f"\nInput: {s2}\n")
print(predict(s2).to_string(index=False))

Input: John works at Google in UK

 Token POS Tag
  John   PROPN
 works    VERB
    at     ADP
Google   PROPN
    in     ADP
    UK   PROPN

Input: The quick brown fox jumps over the lazy dog

Token POS Tag
  The     DET
quick     ADJ
brown     ADJ
  fox    NOUN
jumps    VERB
 over     ADP
  the     DET
 lazy     ADJ
  dog    NOUN
